<a href="https://colab.research.google.com/github/lsgrep/serv/blob/main/notebooks/09_serving_levers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 9 — Serving levers, measured

**The claim you should be able to make when you finish:** *"Every one of those
optimisations, I've turned on and measured. I can tell you what each one buys,
what it costs, and the workload where it does nothing."*

Lab 1's diagnosis playbook *names* these levers. Naming is cheap. This lab turns
each into a number on hardware you have, so that when you recommend one you are
quoting a measurement rather than a README.

| Lever | Claim | Cost |
|---|---|---|
| Prefix caching | huge TTFT win on shared system prompts | cache memory |
| Chunked prefill | fixes p99 TTFT under mixed prompt lengths | small throughput tax |
| FP8 KV cache | halves KV bytes → ~2x concurrency | needs Ada+; small quality risk |
| Speculative decoding | 1.5-2.5x TPOT at high acceptance | hurts at high batch; draft-model ops |
| Tensor parallelism | fits the model, cuts latency | interconnect-bound |

The honest framing to carry into the room: **every one of these is a trade, and
the workload decides.** A candidate who says "enable prefix caching" is repeating
documentation. One who says "prefix caching if the system prompt is a large share
of your input — measure that share first, because on short unique prompts it is
just memory you stopped using for KV" has done it.

In [ ]:
# Cell 1 — bootstrap. Needs a GPU and vLLM; see lab 1 for the install notes.
REPO, BRANCH = "https://github.com/lsgrep/serv.git", "main"

import os, subprocess, sys

if not os.path.isdir("serv"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO], check=True)
else:
    subprocess.run(["git", "-C", "serv", "pull", "--ff-only", "-q"], check=False)
sys.path.insert(0, os.path.abspath("serv"))

def pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

pip("matplotlib", "pandas", "httpx")
pip("vllm")

import servlab
env = servlab.notebook_setup()

## 0. Predict each one before you measure it

Written predictions first. The point of the lab is the gap between these numbers
and the measurements, and you cannot have a gap you did not write down.

In [ ]:
from servlab import napkin as nk

SPEC = nk.MODELS["qwen2.5-3b"]
CARD = "T4" if "T4" in env.gpu_name else ("L4" if "L4" in env.gpu_name else "A100-40GB")
SYS_PROMPT_TOKENS, USER_TOKENS = 800, 200

# Prefix caching: the win is bounded by the share of input you stop re-prefilling.
share = SYS_PROMPT_TOKENS / (SYS_PROMPT_TOKENS + USER_TOKENS)
full = nk.prefill_time_s(CARD, SPEC, SYS_PROMPT_TOKENS + USER_TOKENS)
cached = nk.prefill_time_s(CARD, SPEC, USER_TOKENS)
print(f"prefix caching: system prompt is {share:.0%} of input")
print(f"  prefill {full*1000:.0f} ms -> {cached*1000:.0f} ms   (predicted {full/cached:.1f}x on TTFT)")

# FP8 KV: concurrency should roughly double, if the card supports it.
for kv in ("fp16", "fp8"):
    print(f"KV {kv}: {nk.max_concurrent_sequences(CARD, SPEC, 2048, kv_dtype=kv):,.0f} "
          f"concurrent 2K sequences")
print(f"  this card supports fp8: {env.supports_fp8}")

# Speculative decoding: entirely a function of acceptance rate.
print("\nspeculative decoding, predicted TPOT speedup by acceptance rate:")
for a in (0.9, 0.8, 0.7, 0.6, 0.5, 0.4):
    print(f"  acceptance {a:.0%}: {nk.spec_decode_speedup(a, gamma=4):.2f}x")
print("\n  Below ~0.5 the draft passes cost more than the tokens they save.")

## 1. Prefix caching

The agent-workload lever. Agents send the same long system prompt — tools,
instructions, few-shot examples — on every single turn. Without caching you
re-prefill it every time, and prefill is the compute-bound half.

The measurement that matters is not the speedup; it is **the share of your input
that is shared**. Measure that first, on real traffic, because it decides whether
this lever is worth anything at all.

In [ ]:
from servlab.serve import VLLMServer
from servlab.loadgen import run_load
from servlab.stats import summarize
import gc, time, torch

MODEL = "Qwen/Qwen2.5-3B-Instruct"
BASE = "http://localhost:8000"
SYSTEM = "You are a careful assistant. " * 100   # ~800 tokens of shared prefix

def bench(label, extra=(), concurrency=8, duration=30, prompt_tokens=200,
          max_tokens=64, unique_prompts=True, **serve_kw):
    server = VLLMServer(MODEL, port=8000, max_model_len=2048, gpu_memory_utilization=0.88,
                        enforce_eager=True, extra=list(extra),
                        log_path=f"runs/lab9-{label}.log", **serve_kw).start()
    try:
        run_load(BASE, MODEL, concurrency=2, duration=8, prompt_tokens=prompt_tokens,
                 max_tokens=16, unique_prompts=unique_prompts)          # warm
        t0 = time.perf_counter()
        res = run_load(BASE, MODEL, concurrency=concurrency, duration=duration,
                       prompt_tokens=prompt_tokens, max_tokens=max_tokens,
                       unique_prompts=unique_prompts)
        s = summarize(res, duration=time.perf_counter() - t0, slo_ttft=1.0, slo_tpot=0.05)
        print(f"\n[{label}]\n{s}")
        return s
    finally:
        server.stop()
        gc.collect(); torch.cuda.empty_cache(); time.sleep(5)

In [ ]:
# unique_prompts=False makes every request share a prefix — the agent workload.
no_cache = bench("no-prefix-cache", extra=["--no-enable-prefix-caching"],
                 prompt_tokens=800, unique_prompts=False)

In [ ]:
with_cache = bench("prefix-cache", extra=["--enable-prefix-caching"],
                   prompt_tokens=800, unique_prompts=False)

In [ ]:
from servlab.plots import bar_compare

print(f"TTFT p50: {no_cache.ttft['p50']*1000:,.0f} ms -> {with_cache.ttft['p50']*1000:,.0f} ms "
      f"({no_cache.ttft['p50']/with_cache.ttft['p50']:.2f}x)")
bar_compare(["no prefix cache", "prefix cache"],
            [no_cache.ttft["p50"]*1000, with_cache.ttft["p50"]*1000],
            title="TTFT p50 on a shared-prefix workload", ylabel="ms")

In [ ]:
# The control that makes the claim honest: unique prompts, nothing to share.
unique_cached = bench("prefix-cache-unique-prompts", extra=["--enable-prefix-caching"],
                      prompt_tokens=800, unique_prompts=True)
print(f"\nshared prefixes : {with_cache.ttft['p50']*1000:>8,.0f} ms")
print(f"unique prompts  : {unique_cached.ttft['p50']*1000:>8,.0f} ms")
print("\nSame flag, no benefit — and the cache is consuming memory that would")
print("otherwise hold KV. That control is what turns 'I enabled it' into 'I know")
print("when it helps'.")

## 2. Chunked prefill

The p99 lever. A long prompt occupies a whole prefill step; every decoding
sequence stalls behind it. p50 stays fine, p99 is wrecked, and the queue looks
healthy — which is why this one gets misdiagnosed as a capacity problem.

Chunked prefill splits the prompt across steps so decode interleaves. The long
request's own TTFT gets slightly worse; everyone else's tail gets much better.

Reproduce it with a bimodal workload: mostly short prompts, a few very long ones.

In [ ]:
import threading

def mixed_load(duration=45):
    """Short requests plus a trickle of very long ones — the shape that
    produces head-of-line blocking."""
    out = {}
    def short():
        out["short"] = run_load(BASE, MODEL, rps=3.0, duration=duration,
                                prompt_tokens=128, max_tokens=64)
    def long():
        out["long"] = run_load(BASE, MODEL, rps=0.3, duration=duration,
                               prompt_tokens=1800, max_tokens=64)
    ts = [threading.Thread(target=short), threading.Thread(target=long)]
    [t.start() for t in ts]
    [t.join() for t in ts]
    return out

def mixed_bench(label, extra=()):
    server = VLLMServer(MODEL, port=8000, max_model_len=2048, gpu_memory_utilization=0.88,
                        enforce_eager=True, extra=list(extra),
                        log_path=f"runs/lab9-{label}.log").start()
    try:
        run_load(BASE, MODEL, concurrency=2, duration=8, max_tokens=16)
        out = mixed_load()
        s_short = summarize(out["short"], slo_ttft=1.0)
        s_long = summarize(out["long"], slo_ttft=1.0)
        print(f"\n[{label}]  short requests: p50 {s_short.ttft['p50']*1000:,.0f} ms  "
              f"p99 {s_short.ttft['p99']*1000:,.0f} ms")
        print(f"{'':>{len(label)+3}}  long requests:  p50 {s_long.ttft['p50']*1000:,.0f} ms")
        return s_short, s_long
    finally:
        server.stop()
        gc.collect(); torch.cuda.empty_cache(); time.sleep(5)

In [ ]:
off_short, off_long = mixed_bench("chunked-prefill-off",
                                  extra=["--no-enable-chunked-prefill"])

In [ ]:
on_short, on_long = mixed_bench("chunked-prefill-on",
                                extra=["--enable-chunked-prefill",
                                       "--max-num-batched-tokens", "512"])

In [ ]:
import matplotlib.pyplot as plt
from servlab.plots import SERIES

labels = ["short p50", "short p99", "long p50"]
off = [off_short.ttft["p50"]*1000, off_short.ttft["p99"]*1000, off_long.ttft["p50"]*1000]
on  = [on_short.ttft["p50"]*1000,  on_short.ttft["p99"]*1000,  on_long.ttft["p50"]*1000]

fig, ax = plt.subplots(figsize=(7.5, 4.2))
x = range(len(labels))
ax.bar([i-0.19 for i in x], off, width=0.36, color=SERIES[1], label="chunked prefill off")
ax.bar([i+0.19 for i in x], on,  width=0.36, color=SERIES[0], label="chunked prefill on")
ax.set_xticks(list(x)); ax.set_xticklabels(labels)
ax.set_ylabel("TTFT (ms)")
ax.set_title("chunked prefill: the tail improves, the long request pays for it")
ax.legend()
plt.show()

print("The trade in one line: short-request p99 improves, the long request's own")
print("TTFT gets slightly worse, and aggregate throughput takes a small tax.")
print("Whether that is a good deal depends on who your SLA is written for.")

## 3. FP8 KV cache

Halves KV bytes per token, so roughly doubles the sequences the card can hold.
On a T4 this cell will tell you why it cannot run — FP8 needs Ada (sm_89) or
newer, and knowing that boundary is itself the answer to a common question.

Always gate a quantisation decision on **your** eval, never the paper's. Lab 5
has the harness for that; this is only the capacity half.

In [ ]:
if not env.supports_fp8:
    print(f"{env.gpu_name} is sm_{env.capability[0]}{env.capability[1]} — FP8 KV needs sm_89+.")
    print("The prediction stands and is worth quoting: halving KV bytes per token")
    print("roughly doubles concurrency at a given context length.")
    print(f"  fp16 KV: {nk.max_concurrent_sequences(CARD, SPEC, 2048, kv_dtype='fp16'):,.0f} seqs")
    print(f"  fp8  KV: {nk.max_concurrent_sequences(CARD, SPEC, 2048, kv_dtype='fp8'):,.0f} seqs")
else:
    fp16_kv = bench("kv-fp16", concurrency=32, duration=30)
    fp8_kv = bench("kv-fp8", extra=["--kv-cache-dtype", "fp8"], concurrency=32, duration=30)
    print(f"\nthroughput: {fp16_kv.output_throughput:,.0f} -> "
          f"{fp8_kv.output_throughput:,.0f} out tok/s")

In [ ]:
# Whichever branch ran, read what the engine actually allocated. The log line is
# ground truth; the napkin math was the prediction.
print("\n".join(l for l in open("runs/lab9-kv-fp16.log", errors="replace").read().splitlines()
                 if "KV cache" in l or "concurrency" in l)[:600]
      if os.path.exists("runs/lab9-kv-fp16.log") else "(run the cell above first)")

## 4. Speculative decoding

A small draft model proposes `gamma` tokens; the target verifies all of them in
one forward pass. When the draft is right, you got several tokens for the price
of one.

Two things to say, and they are what separate having read about it from having
run it:

1. **It is a latency optimisation that spends spare compute.** Decode at low
   batch leaves the GPU mostly idle, and speculation fills that. At high batch
   there is no spare compute, so the win shrinks — sometimes to nothing.
2. **The acceptance rate is everything.** Below roughly 0.6 the draft passes cost
   more than the tokens they save. The draft must match the target's
   *distribution*, not merely be small — which is why same-family drafts work
   and generic tiny models often do not.

In [ ]:
import matplotlib.pyplot as plt
from servlab.plots import SERIES, STATUS

acceptances = [i / 20 for i in range(2, 20)]
fig, ax = plt.subplots(figsize=(7.5, 4.2))
for i, gamma in enumerate((2, 4, 8)):
    ax.plot([a * 100 for a in acceptances],
            [nk.spec_decode_speedup(a, gamma=gamma) for a in acceptances],
            color=SERIES[i], label=f"gamma={gamma}")
ax.axhline(1.0, color=STATUS["critical"], linestyle=":", linewidth=1.5)
ax.annotate("break-even", xy=(20, 1.0), xytext=(0, 5), textcoords="offset points",
            color=STATUS["critical"], fontsize=9)
ax.set_xlabel("acceptance rate (%)"); ax.set_ylabel("TPOT speedup")
ax.set_title("speculative decoding lives or dies on acceptance")
ax.legend()
plt.show()

print("Longer speculation (higher gamma) wins more when acceptance is high and")
print("loses more when it is low — it is a leveraged bet on the draft model.")

In [ ]:
# Measure it, if there is room for two models on this card. On a 16 GB T4 with a
# 3B target there usually is not, and saying so is the right answer.
DRAFT = "Qwen/Qwen2.5-0.5B-Instruct"
if env.vram_gb >= 22:
    base = bench("no-spec", concurrency=1, duration=30, max_tokens=128)
    spec = bench("spec-decode", concurrency=1, duration=30, max_tokens=128,
                 extra=["--speculative-model", DRAFT, "--num-speculative-tokens", "4"])
    print(f"\nTPOT: {base.tpot['p50']*1000:.1f} ms -> {spec.tpot['p50']*1000:.1f} ms "
          f"({base.tpot['p50']/spec.tpot['p50']:.2f}x at batch 1)")
    high_batch = bench("spec-decode-batch32", concurrency=32, duration=30, max_tokens=128,
                       extra=["--speculative-model", DRAFT, "--num-speculative-tokens", "4"])
    print("Now compare the batch-32 number: the win should be much smaller, because")
    print("there is no spare compute left for the draft model to use.")
else:
    print(f"{env.vram_gb:.0f} GB is not enough for a 3B target plus a draft model plus KV.")
    print("The honest answer in an interview: 'speculative decoding needs headroom for")
    print("two models — on a 16 GB card I'd use a smaller target or a larger GPU, and")
    print("I'd expect 1.5-2x on TPOT at batch 1 and much less at production batch sizes.'")

## 5. Tensor parallelism

TP splits the weights across cards, so the memory term divides — but it adds an
all-reduce after every attention and MLP block, twice per layer. Whether it
helps is a comparison between those two terms, not a property of TP.

In [ ]:
rows = nk.tp_scaling("H100-80GB", "llama-3.3-70b", batch=1, ctx_len=4096, weight_dtype="fp8")
print("70B fp8 on H100s, NVLink (6 us/layer all-reduce):")
for r in rows:
    print(f"  tp={r['tp']:>2}  {r['step_s']*1000:>6.2f} ms  "
          f"{r['speedup']:>5.2f}x  efficiency {r['efficiency']:>5.0%}")

slow = nk.tp_scaling("H100-80GB", "llama-3.3-70b", batch=1, ctx_len=4096,
                     weight_dtype="fp8", allreduce_us_per_layer=60)
print("\nsame thing off NVLink (60 us/layer):")
for r in slow:
    print(f"  tp={r['tp']:>2}  {r['step_s']*1000:>6.2f} ms  "
          f"{r['speedup']:>5.2f}x  efficiency {r['efficiency']:>5.0%}")

small_fast = nk.decode_step_time_tp_s("H100-80GB", "llama-3.2-1b", tp=1)
small_slow = nk.decode_step_time_tp_s("H100-80GB", "llama-3.2-1b", tp=8,
                                      allreduce_us_per_layer=60)
print(f"\nand a 1B model 8-way on a slow fabric: {small_fast*1000:.2f} ms -> "
      f"{small_slow*1000:.2f} ms — TP made it slower.")
print("TP pays while the memory term dominates the communication term. That is the")
print("whole rule, and it is why 'just use TP=8' is not advice.")

## 6. The summary you would actually give a customer

Fill this in from your own measurements. The value is not the table — it is
having numbers you took yourself, on hardware you can name.

| Lever | Measured here | Turn it on when | Skip it when |
|---|---|---|---|
| Prefix caching | `[your TTFT ratio]` | the shared prefix is a large share of input | prompts are unique — it is just memory |
| Chunked prefill | `[your p99 delta]` | prompt lengths are bimodal | uniform short prompts |
| FP8 KV | `[your concurrency delta]` | memory-bound on Ada+ and the eval passes | quality-critical, or pre-Ada |
| Speculative decoding | `[your TPOT ratio]` | latency product, low batch, good draft | throughput fleet at high batch |
| Tensor parallelism | `[your scaling]` | the model does not fit, or latency SLA | small model, or off NVLink |

## What to be able to say afterwards

1. **Each lever's trade, in one sentence**, plus the workload where it does
   nothing — that second half is what shows you have measured rather than read.
2. **Why speculative decoding fades at high batch** (no spare compute to spend).
3. **Why chunked prefill is a p99 fix, not a throughput fix**, and who pays for it.
4. **The prefix-cache question to ask first**: what share of input is actually
   shared? Measure before enabling.
5. **When TP is a trap**: small memory term, slow interconnect.

**Back to:** [lab 1](01_serving_under_load.ipynb), whose diagnosis playbook now
has measurements behind every recommendation.